# Round 1: ellipsoid on AoU's premade PCs around a 1000G superpopulation (`superpop`)

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

## Functions

In [ ]:
import ast

def parse_pcs(col, n=16):
    arr = np.vstack(col.map(ast.literal_eval).to_numpy())
    assert arr.shape[1] == n, arr.shape
    return arr

def read_1kg_labels(path):
    kg = pd.read_csv(path, sep=r"\s+", dtype=str)
    return pd.DataFrame({"s": kg["SampleID"], "pop": kg["Population"], "region": kg["Superpopulation"],
                         "founder": (kg["FatherID"] == "0") & (kg["MotherID"] == "0")})
def mahalanobis(Z, mu, C):
    d = Z[:, :len(mu)] - mu
    return np.sqrt(np.einsum("ij,jk,ik->i", d, np.linalg.inv(C), d))

def gate_ellipse(ax, mu, C, i, j, r, **kw):
    """Outline of the ellipsoid on PCs i, j."""
    from matplotlib.patches import Ellipse
    vals, vecs = np.linalg.eigh(C[np.ix_([i, j], [i, j])])
    ax.add_patch(Ellipse((mu[i], mu[j]), 2 * r * np.sqrt(vals[-1]), 2 * r * np.sqrt(vals[0]),
                         angle=np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1])), fill=False, **kw))

def fit_quantile_gate(R, width):
    """MVN on reference founders; the radius holds `width` of them."""
    mu, C = R.mean(0), np.cov(R, rowvar=False)
    ref_d = mahalanobis(R, mu, C)
    return {"mu": mu, "C": C, "radius": float(np.quantile(ref_d, width)), "ref_d": ref_d}

def coverage_curve(dist, ref_d, cover=np.linspace(0.05, 0.999, 60)):
    radii = np.quantile(ref_d, cover)
    return pd.DataFrame({"coverage": cover, "kept": [int((dist <= r).sum()) for r in radii]})

def hexbin_safe(ax, x, y, **kw):
    """Participant density; cells with fewer than 21 people are not drawn."""
    return ax.hexbin(x, y, **{"gridsize": 120, "bins": "log", "cmap": "Greys", **kw, "mincnt": 21})

## Load

In [ ]:
STAGE = "ancestry_round1"
require_ready(cfg, STAGE)
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
SUPERPOP, N_PCS, WIDTH = cfg.param("superpop"), cfg.param("r1_n_pcs"), cfg.param("r1_width")

aou = pd.read_csv(cfg.input("ancestry_preds"), sep="\t", usecols=["research_id", "pca_features"],
                  dtype={"research_id": str})
X = parse_pcs(aou.pop("pca_features"))
ref = pd.read_csv(cfg.input("training_pca"), sep="\t", usecols=["s", "scores"], dtype={"s": str})
R = parse_pcs(ref.pop("scores"))
ref = ref.merge(read_1kg_labels(cfg.input("ref_labels")), on="s", how="left", validate="one_to_one")
REF_POPS = sorted(ref.loc[ref["region"].eq(SUPERPOP), "pop"].unique())
rows = (ref["pop"].isin(REF_POPS) & ref["founder"].eq(True)).to_numpy()

rep.check(f"{SUPERPOP} populations found in the reference", len(REF_POPS) > 0, ", ".join(REF_POPS))
rep.check("gate fit on founders only", True,
          f"{int((ref['pop'].isin(REF_POPS) & ref['founder'].eq(False)).sum())} non-founders left out")
rep.check("enough founders for the covariance", rows.sum() > 10 * N_PCS, f"{rows.sum()} founders, {N_PCS} PCs")

## Gate

In [ ]:
g = fit_quantile_gate(R[rows, :N_PCS], WIDTH)
d = mahalanobis(X, g["mu"], g["C"])
keep = d <= g["radius"]
rep.note(superpop=SUPERPOP, ref_pops=",".join(REF_POPS), n_pcs=N_PCS, width=WIDTH, radius=round(g["radius"], 3))
rep.count("n_participants", len(X))
print("kept", rep.count("n_kept", keep.sum()))

In [ ]:
cc = coverage_curve(d, g["ref_d"])
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(cc["coverage"] * 100, cc["kept"].where(cc["kept"] > 20))
ax.axvline(WIDTH * 100, color="0.6", ls=":")
ax.set(yscale="log", xlabel="reference founders inside (%)", ylabel="participants inside")
rep.fig(fig, "kept_vs_coverage", "plateau = cluster, steady growth = continuum"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (i, j) in zip(axes, [(0, 1), (2, 3)]):
    hexbin_safe(ax, X[:, i], X[:, j])
    for p in REF_POPS:
        m = rows & ref["pop"].eq(p).to_numpy()
        ax.scatter(R[m, i], R[m, j], s=8, marker="x", label=p)
    gate_ellipse(ax, g["mu"], g["C"], i, j, g["radius"], edgecolor="tab:red", lw=1.5)
    ax.set(xlabel=f"PC{i+1}", ylabel=f"PC{j+1}")
axes[0].legend(fontsize=8)
rep.fig(fig, "gate_pcs"); plt.show()

In [ ]:
founders = ref[ref["founder"].eq(True) & ref["pop"].notna()]
inside = (founders.assign(inside=mahalanobis(R[founders.index], g["mu"], g["C"]) <= g["radius"])
          .groupby(["region", "pop"])["inside"].mean().mul(100).round(1).reset_index(name="pct_inside"))
rep.table(inside, "reference_inside_gate")

In [ ]:
rel = pd.read_csv(cfg.input("relatedness"), sep="\t", dtype={"i.s": str, "j.s": str})
ids = set(aou.loc[keep, "research_id"])
both = rel["i.s"].isin(ids) & rel["j.s"].isin(ids)
cls = pd.cut(rel.loc[both, "kin"], [0.1, 0.177, 0.354, 1.01], right=False,
             labels=["2nd degree", "1st degree", "dup/MZ"])
kin = cls.value_counts().sort_index().rename_axis("class").reset_index(name="pairs")
rep.table(kin, "related_pairs_inside", counts=["pairs"])

## Write

In [ ]:
if needs_run(OUT, FORCE):
    with record(cfg, STAGE, OUT) as meta:
        get_store(OUT).write_text(OUT, "\n".join(aou.loc[keep, "research_id"]) + "\n")
        meta.update(n_kept=int(keep.sum()), radius=g["radius"])
rep.save()